In [1]:
import openai
from qdrant_client import QdrantClient

### Embedding function

In [2]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input = text,
        model = model,
    )
    return response.data[0].embedding

### Retrieval function

In [4]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [48]:
def retrieval_data(query, qdrant_client, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name = "Amazon-items-collection-00",
        query = query_embedding,
        limit = k,
    )
    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_score": similarity_scores,
    }

In [49]:
retrieved_context = retrieval_data("What kind of earphones can I get?", qdrant_client, k=10)

In [15]:
retrieved_context

{'retrieved_context_ids': ['B0CBMPG524',
  'B0B9FTVL58',
  'B0C142QS8X',
  'B0C6K1GQCF',
  'B0BNHVLF7G',
  'B0B14HTZ59',
  'B09WCFC5D9',
  'B0B67ZFRPC',
  'B0B7495RL6',
  'B0BS15TRJ3'],
 'retrieved_context': ['Open Ear Headphones, Bluetooth 5.3 Earbuds with 60H Playtime IPX7 Waterproof Wireless Earbuds Immersive Premium Sound True Wireless Open Ear Earbuds with Earhooks for Running, Walking and Workouts 【Open-ear Design Headphones】Feature with a new generation of true open-ear wireless earbuds design, the headphones can rest gently and firmly fit your ears without entering your ear canal, which will reduce stress and hearing loss after extended wear. There is no pinching of the auricle, no blockage of the ear canal, and no pain or damage to hearing. 【Powerful Stereo Sound】Equipped with 16.2 millimeters vibrating diaphragm speaker driver, bluetooth headphones providing pure balanced audio and clarity output for all music genres with soft audio and immerse yourself in the wonderful world

### Format retrieved context function  

To make it a more readable format before passing the RAG retrieved context to LLM

In [23]:
def process_context(context):
    formatted_context = ""

    for id, chunk in zip(context["retrieved_context_ids"], context["retrieved_context"]):
        formatted_context += f"- {id}: {chunk}\n"

    return formatted_context

In [34]:
preprocessed_context = process_context(retrieved_context)

In [41]:
print(preprocessed_context)

- B0CBMPG524: Open Ear Headphones, Bluetooth 5.3 Earbuds with 60H Playtime IPX7 Waterproof Wireless Earbuds Immersive Premium Sound True Wireless Open Ear Earbuds with Earhooks for Running, Walking and Workouts 【Open-ear Design Headphones】Feature with a new generation of true open-ear wireless earbuds design, the headphones can rest gently and firmly fit your ears without entering your ear canal, which will reduce stress and hearing loss after extended wear. There is no pinching of the auricle, no blockage of the ear canal, and no pain or damage to hearing. 【Powerful Stereo Sound】Equipped with 16.2 millimeters vibrating diaphragm speaker driver, bluetooth headphones providing pure balanced audio and clarity output for all music genres with soft audio and immerse yourself in the wonderful world of music. 【Hear Your Surroundings】Open earbuds that rest on your ears without covering them, you can hear your music and your surroundings at the same time. Whether you are cycling, walking, runn

### Create Prompt function

In [ ]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""

    return prompt

In [44]:
prompt = build_prompt(preprocessed_context, "What kind of earphones can I get?")

In [40]:
print(build_prompt(preprocessed_context, "What kind of earphones can I get?"))


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use word context and refer to it as the available products.

Context:
- B0CBMPG524: Open Ear Headphones, Bluetooth 5.3 Earbuds with 60H Playtime IPX7 Waterproof Wireless Earbuds Immersive Premium Sound True Wireless Open Ear Earbuds with Earhooks for Running, Walking and Workouts 【Open-ear Design Headphones】Feature with a new generation of true open-ear wireless earbuds design, the headphones can rest gently and firmly fit your ears without entering your ear canal, which will reduce stress and hearing loss after extended wear. There is no pinching of the auricle, no blockage of the ear canal, and no pain or damage to hearing. 【Powerful Stereo Sound】Equipped with 16.2 millimeters vibrating diaphragm speaker driver, bluetooth headphones providing pure balanced 

### Generate answer function

In [42]:
def generate_answer(prompt):

    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = [{"role": "system", "content": prompt}],
        temperature = 0.5
    )
    return response.choices[0].message.content

In [47]:
print(generate_answer(prompt))

You can get several kinds of earphones from the available products:

1. Open Ear Headphones: Bluetooth 5.3 earbuds with an open-ear design that rest gently on your ears without entering the ear canal. They provide immersive premium sound, 60 hours playtime with charging case, IPX7 waterproof, and are suitable for running, walking, and workouts (B0CBMPG524).

2. Wireless In-Ear Earbuds: Bluetooth 5.3 headphones with microphone, 37 hours playback, LED power display, deep bass, IPX7 waterproof, ultra-light, and smart touch controls (B0B9FTVL58).

3. Kids Over Ear Headphones: Volume limited to 94dB for hearing protection, foldable and adjustable with soft ear cups, compatible with 3.5mm audio jack devices, designed for kids (B0C142QS8X).

4. Wireless Earbuds with Noise Cancelling Mic: Bluetooth 5.1 earbuds with 30 hours playtime, IPX7 waterproof, ergonomic design, touch control, suitable for iPhone and Android (B0C6K1GQCF).

5. Bone Conduction Headphones: Wireless Bluetooth 5.3 open-ear sp

### Combined RAG Pipeline

In [50]:
def rag_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url="http://localhost:6333")
    
    retrieved_context = retrieval_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)   
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return answer

In [52]:
print(rag_pipeline("Can I get some cool stuff for kids?"))

Yes, you can get some cool stuff for kids from the available products:

1. TUNEAKE Kids Headphones (Pink) - Over ear, volume limited to 94dB for hearing protection, foldable and adjustable for comfort, compatible with many devices, great for school, travel, and play.

2. QearFun Cat Earbuds for Kids (Blue) - Wired earbuds with a microphone and a lovely earphones storage case, perfect for school girls and boys.

These items are both fun and practical for kids.
